# RoI Align

RoI Align 将每个候选框映射为固定大小的特征网格，常用于两阶段目标检测与实例分割。与 RoI Pooling 不同，它不对边界或采样点坐标取整，因此避免量化误差。

设缩放后的 RoI 为 $(x_1,y_1,x_2,y_2)$，输出大小为 $H_o\times W_o$，则每个 bin 的尺寸为：

$$\Delta x=\frac{x_2-x_1}{W_o},\qquad \Delta y=\frac{y_2-y_1}{H_o}.$$

对于连续坐标 $(x,y)$，以其相邻四个特征值 $(v_{00},v_{01},v_{10},v_{11})$ 做双线性插值：

$$v(x,y)=(1-d_x)(1-d_y)v_{00}+(1-d_x)d_yv_{01}+d_x(1-d_y)v_{10}+d_xd_yv_{11}.$$

其中 $d_x=x-\lfloor x\rfloor$、$d_y=y-\lfloor y\rfloor$。每个 bin 对均匀采样点的插值结果取平均；本实现为每个 bin 使用中心采样点。

In [ ]:
import torch
import torch.nn.functional as F

def bilinear_interpolate(feature_map, x, y):
    """
    在单个连续坐标 (x, y) 上采样特征图。
    feature_map: [N, C, H, W]；本实现返回 [N, C]，保留批次和通道维。
    x, y: 标量浮点坐标；x 对应宽度 W，y 对应高度 H。
    """
    # 左上整数网格坐标；floor 后是 Python 标量，便于用于张量索引。
    x0, y0 = int(torch.floor(x)), int(torch.floor(y))
    x1, y1 = x0 + 1, y0 + 1
    
    # 将四个邻点限制在有效空间范围：[0, W - 1] 与 [0, H - 1]。
    x0 = max(0, min(x0, feature_map.shape[3] - 1))
    x1 = max(0, min(x1, feature_map.shape[3] - 1))
    y0 = max(0, min(y0, feature_map.shape[2] - 1))
    y1 = max(0, min(y1, feature_map.shape[2] - 1))
    
    # 计算连续位置距左/上邻点的小数距离；四个权重之和为 1。
    wx1, wy1 = x - x0, y - y0
    wx0, wy0 = 1 - wx1, 1 - wy1
    
    # 高级索引固定空间坐标，保留前两维，因此每个角点张量形状均为 [N, C]。
    v00 = feature_map[:, :, y0, x0]
    v01 = feature_map[:, :, y1, x0]
    v10 = feature_map[:, :, y0, x1]
    v11 = feature_map[:, :, y1, x1]
    
    # 加权和仍为 [N, C]；这里由调用方写入某个输出 RoI 的 [C] 切片。
    return v00 * wy0 * wx0 + v01 * wy1 * wx0 + v10 * wy0 * wx1 + v11 * wy1 * wx1

def roi_align(feature_map, rois, output_size, spatial_scale, sampling_ratio=2):
    """
    feature_map: [1, C, H, W]，本教学实现仅支持 batch size 为 1。
    rois: [K, 5]，每行是 (batch_idx, x1, y1, x2, y2)。
    output_size: (out_h, out_w)，每个 RoI 被规整到的空间尺寸。
    spatial_scale: 原图坐标到特征图坐标的缩放比例。
    sampling_ratio: 每个 bin 的采样点数；当前简化实现固定使用中心点。
    return: [K, C, out_h, out_w]。
    """
    out_h, out_w = output_size
    num_rois = rois.shape[0]
    C = feature_map.shape[1]
    # 预分配输出；每个 RoI 的 C 个通道均生成 out_h × out_w 个采样值。
    output = torch.zeros(num_rois, C, out_h, out_w, device=feature_map.device)  # [K, C, out_h, out_w]
    
    for i in range(num_rois):
        # 1. 从第 i 行 [5] 中读取连续坐标并缩放；全程不取整。
        x1 = rois[i, 1] * spatial_scale
        y1 = rois[i, 2] * spatial_scale
        x2 = rois[i, 3] * spatial_scale
        y2 = rois[i, 4] * spatial_scale
        
        # 这两个标量描述当前 RoI 在特征图坐标系中的宽、高。
        roi_w = x2 - x1
        roi_h = y2 - y1
        
        bin_w = roi_w / out_w
        bin_h = roi_h / out_h
        
        # 2. 遍历输出网格；ph/pw 是输出张量最后两维 [out_h, out_w] 的索引。
        for ph in range(out_h):
            for pw in range(out_w):
                # 当前 bin 的连续边界；每个 bin 的大小都是 bin_h × bin_w。
                bin_x1 = x1 + pw * bin_w
                bin_y1 = y1 + ph * bin_h
                bin_x2 = bin_x1 + bin_w
                bin_y2 = bin_y1 + bin_h
                
                # 3. 在 bin 内均匀采样
                # 这里简化采样，实际通常每个 bin 采样 2x2=4个点
                sample_x = (bin_x1 + bin_x2) / 2
                sample_y = (bin_y1 + bin_y2) / 2
                
                # 4. 插值结果为 [1, C]；索引 [0] 后变为 [C]，与目标切片精确对齐。
                output[i, :, ph, pw] = bilinear_interpolate(feature_map, sample_x, sample_y)[0]
                
    return output